## Initialization

### Imports

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import make_interp_spline
from pint import Quantity

from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
from data_processing import helpers
# from data_processing.paths import (
#     get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    # NonReactorDataframeColumn,
    # SliceFitDataframeColumn,
    EnergyColumn,
    get_df_col
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
# from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing.processing.neutron_window_strategy.strategy_factory \
    import NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy \
    import AbstractNeutronStrategy
# from data_processing.helpers import (
#     # get_input_with_default,
#     # get_input_required,
#     # input_experiment_ids,
#     stop,
#     get_midpoints_from_bins
# )


### Functions

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> proc_types.NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = proc_types.NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data


In [ ]:
# def relative_rmse(x: pd.Series | float, x_err: pd.Series | float, y: pd.Series | float, y_err: pd.Series | float) -> pd.Series | float:
def relative_rmse(values: list[tuple[pd.Series | float, pd.Series | float]]) -> pd.Series | float:
    rel_sq_values = [relative_square_error(x, x_err) for x, x_err in values]
    # rel_sq_x = relative_square_error(x, x_err)
    # rel_sq_y = relative_square_error(y, y_err)
    # rel_sq_sum = rel_sq_x + rel_sq_y
    rel_sq_sum = sum(rel_sq_values)
    if isinstance(rel_sq_sum, pd.Series):
        return rel_sq_sum.pow(1./2)
    else:
        return rel_sq_sum ** (1./2)


def relative_square_error(x: pd.Series | float, x_err: pd.Series | float) -> pd.Series | float:
    # divide x_err by x
    # square it
    # return
    rel_err = x_err / x
    if isinstance(rel_err, pd.Series):
        return rel_err.pow(2).fillna(0)
    else:
        return rel_err ** 2

In [ ]:
def correct_raw_signals(
    raw_signals_df: pd.DataFrame,
    baseline_idx_range: int = 40,
    baseline_offset: float = 0,
    max_adc: int = 16367,
    use_max_adc: bool = False
) -> pd.DataFrame:
    offset = int(baseline_offset * max_adc)
    signals_np = raw_signals_df.to_numpy()
    
    if use_max_adc:
        baselines = max_adc
    else:
        baselines = signals_np[
            :, :baseline_idx_range
        ].mean(axis=1).reshape(-1, 1)
    
    signals_np = -signals_np + baselines + offset
    corrected_signals = pd.DataFrame(
        signals_np,
        index=raw_signals_df.index,
        columns=raw_signals_df.columns
    )
    return corrected_signals

In [ ]:
def integrate_pulses(pulse_data: np.array, t_start: int, t_end: int):
    left_vals = pulse_data[:, t_start : t_end]
    right_vals = pulse_data[:, t_start+1 : t_end+1]
    # left_vals = pulse_series.loc[t_start:t_end]
    # right_vals = pulse_series.loc[t_start+1:t_end+1]
    
    print(left_vals.shape, right_vals.shape)
    print(left_vals)
    print(right_vals)
    midpoints = (left_vals + right_vals) / 2
    print(midpoints)
    column_areas = midpoints * 2  ## 2 ns between data points
    print(column_areas)
    areas = column_areas.sum(axis=1)
    return areas

## Data Loading

### Loading Params

In [ ]:
experiment_ids = ["TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

In [ ]:
# experiment_ids = input_experiment_ids()

In [ ]:
# # more here?
# experiment_neutron_data: ExperimentNeutronData = {
#     exp_id: {}
#     for exp_id in experiment_ids
# }

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )

# is_new_calibration = calib_input.lower() == "y"
# calibrated_energy_column: EnergyColumn = (
#     DetectorDataframeColumn.RECALIBRATED_ENERGY
#     if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
# )
# calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
# strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
# factory_fn = make_strategy_factory_fn(
#     strategy_factory, "nasa", False, settings)
# experiment_neutron_data = make_strategy_for_experiments(
#     experiment_neutron_data, factory_fn)

### Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
# Ensure that index matches between signals and CAEN data
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    signals_df = exp_data["signals_df"]
    
    unclassified_index: pd.Index = unclassified_df.index
    signals_index: pd.Index = signals_df.index
    clean_index = unclassified_index.intersection(signals_index)
    
    unclassified_df = unclassified_df.loc[clean_index]
    signals_df = signals_df.loc[clean_index]
    
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df
    exp_data["signals_df"] = signals_df

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Data Processing

### Pulse Processing

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    signals_df = exp_data["signals_df"].astype("int32")
    
    signals_df = correct_raw_signals(signals_df)
    heights = signals_df.max(axis=1)
    signals_df.columns = signals_df.columns.map(int)
    
    psd_report["height"] = heights
    exp_data["signals_df"] = signals_df
    exp_data[ExperimentDataKey.UNCLASSIFIED] = psd_report

### Integration

In [ ]:
trigger_time = 144
pre_gate = 50
short_gate = 22
gate_width = 250
baseline_time = 90
t1 = trigger_time - pre_gate
t2 = t1 + short_gate
t3 = t1 + gate_width
print(t1, t2, t3)

In [ ]:
trigger_idx = 72
pre_gate_idx = 25
short_gate_idx = 11
gate_width_idx = 125
idx1 = trigger_idx - pre_gate_idx
idx2 = idx1 + short_gate_idx
idx3 = idx1 + gate_width_idx
print(idx1, idx2, idx3)

In [ ]:
testarray = np.array(
    [[1, 2, 3],
     [2, 3, 4],
     [3, 4, 5]]
)
integ_result = integrate_pulses(testarray, 0, 2)
print(integ_result)

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    signals_df = exp_data["signals_df"]
    signals_np = signals_df.values

    total_areas = integrate_pulses(signals_np, idx1, idx3)
    tail_areas = integrate_pulses(signals_np, idx2, idx3)
    # print(total_areas)
    # print(tail_areas)
    psd_report["new_tail"] = tail_areas
    psd_report["new_total"] = total_areas
    psd_report["new_psd"] = tail_areas / total_areas

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    tail_areas = psd_report["new_tail"]
    total_areas = psd_report["new_total"]

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    old_psd = psd_report["tail / total"]
    new_psd = psd_report["new_psd"]

    fig, ax = plt.subplots()
    ax.scatter(old_psd.values, new_psd.values)
    # ax.set_ylim(0, 1)
    # ax.set_xlim(0, 0.5)

### Neutron Classification

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = proc.get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    helpers.stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

### Neutron/Gamma Separation

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    
    gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
    neutrons_only = psd_report.query(n_class_col_name).copy()
    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
    signals_df = exp_data["signals_df"]
    
    neutron_signals = signals_df.loc[neutrons_only.index]
    gamma_signals = signals_df.loc[gamma_only.index]

    exp_data["neutron_signals"] = neutron_signals
    exp_data["gamma_signals"] = gamma_signals

### Figure 7a/b Processing

In [ ]:
exp_id = "TB-26"

In [ ]:
min_height = 10000
max_height = 16000
height_tolerance = 0.05
psd_separation = 0.175

exp_data = experiment_neutron_data[exp_id]
neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
neutron_signals = exp_data["neutron_signals"]
gamma_signals = exp_data["gamma_signals"]

neutrons_only["PSD"] = neutrons_only["tail / total"]
gamma_only["PSD"] = gamma_only["tail / total"]

selected_neutron = None
selected_gamma = None

bad_neutrons = [64413, 67314, 125305, 127752, 588996, 2168883]
bad_gamma = [2208854]

# # TODO figure out how itertuples works!
# print(neutrons_only.head())
# for i, neutron_tuple in enumerate(neutrons_only.itertuples()):
#     if i > 5:
#         break
#     print(i)
#     print(neutron_tuple.Index, neutron_tuple.PSD, neutron_tuple.height)

for neutron_tuple in neutrons_only.itertuples():
    # get clean neutron pulse (no secondary peak)
    neutron_id = neutron_tuple.Index
    n_height = neutron_tuple.height
    n_psd = neutron_tuple.PSD
    if selected_neutron is not None:
        break
    if neutron_id in bad_neutrons:
        continue
    # n_height = neutron_signal.max()
    if n_height < min_height or n_height > max_height:
        continue
    # peaks, peak_data = find_peaks(neutron_signal, height=200, prominence=50)
    # filtered_peaks = [peak for peak in peaks if abs(peak - 50) > 15]
    # if len(filtered_peaks) == 0:
    else:
        # get matching clean gamma pulse
        for gamma_tuple in gamma_only.itertuples():
            gamma_id = gamma_tuple.Index
            g_height = gamma_tuple.height
            g_psd = gamma_tuple.PSD
            if selected_gamma is not None:
                break
            if gamma_id in bad_gamma:
                continue
            # g_height = gamma_signal.max()
            if abs(g_height - n_height) > (height_tolerance * n_height):
                continue
            if abs(n_psd - g_psd) < psd_separation:
                continue
            # peaks, peak_data = find_peaks(gamma_signal, height=200, prominence=50)
            # filtered_peaks = [peak for peak in peaks if abs(peak - 50) > 15]
            # if len(filtered_peaks) > 0:
            #     continue
            print(f"Neutron ID = {neutron_id}, height = {n_height}, PSD = {n_psd: .3f}")
            print(f"Gamma ID = {gamma_id}, height = {g_height}, PSD = {g_psd: .3f}")
            print(f"PSD delta = {n_psd - g_psd:.3f}")
            selected_neutron = neutron_id, neutron_signals.loc[neutron_id]
            selected_gamma = gamma_id, gamma_signals.loc[gamma_id]

if selected_neutron is None or selected_gamma is None:
    raise Exception("No pulse found")
exp_data["selected_neutron"] = selected_neutron
exp_data["selected_gamma"] = selected_gamma

## Plotting

### Plot Functions

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"
bg_green = "#21d894"

In [ ]:
def plot_figure_7a(ax: mpl.axes.Axes):
    # neutron/gamma pulses, y axis log scale
    exp_data = experiment_neutron_data["TB-26"]
    neutron_id, selected_neutron = exp_data["selected_neutron"]
    gamma_id, selected_gamma = exp_data["selected_gamma"]

    neutron_y = selected_neutron.values.copy()
    gamma_y = selected_gamma.values.copy()
    neutron_x = np.arange(0, len(neutron_y)) * 2
    gamma_x = np.arange(0, len(gamma_y)) * 2

    log_adjust = np.min((neutron_y.min(), gamma_y.min()))
    neutron_y = neutron_y - log_adjust + 1
    gamma_y = gamma_y - log_adjust + 1

    ax.fill_between(neutron_x, neutron_y, color=bg_blue, lw=3)
    ax.fill_between(gamma_x, gamma_y, color=bg_bluegrey, lw=3)
    ax.set_yscale("log")
    
    ax.tick_params(labelsize=fontsize)
    ax.set_xlabel("Time (ns)", fontsize=fontsize)
    ax.set_ylabel("Pulse height (ADC channel)", fontsize=fontsize)

In [ ]:
def plot_figure_7b(ax: mpl.axes.Axes, baseline_time: int, t1: int, t2: int, t3: int, baseline_adjust: float, tail_base: float):
    exp_data = experiment_neutron_data["TB-26"]
    neutron_id, selected_neutron = exp_data["selected_neutron"]
    gamma_id, selected_gamma = exp_data["selected_gamma"]

    base_neutron_y = selected_neutron.values
    base_gamma_y = selected_gamma.values
    base_neutron_x = np.arange(0, len(base_neutron_y)) * 2
    base_gamma_x = np.arange(0, len(base_gamma_y)) * 2
    
    neutron_x = np.linspace(0, base_neutron_x[-1], num=1000)
    gamma_x = np.linspace(0, base_gamma_x[-1], num=1000)
    neutron_y = np.interp(neutron_x, base_neutron_x, base_neutron_y)
    gamma_y = np.interp(gamma_x, base_gamma_x, base_gamma_y)

    where_baseline = neutron_x <= baseline_time
    where_total = (neutron_x > t1) & (neutron_x <= t3)
    where_tail = (neutron_x > t2) & (neutron_x <= t3)

    baseline_x = baseline_time / 2
    baseline_y = -(baseline_adjust / 2)
    total_x = (t1 + t3) / 2
    total_y = -((baseline_adjust + tail_base) / 2)
    tail_x = (t2 + t3) / 2
    tail_y = -(tail_base / 2)

    ax.fill_between(neutron_x, neutron_y, -baseline_adjust, where=where_baseline, color="lightgrey")
    ax.fill_between(neutron_x, neutron_y, -baseline_adjust, where=where_total, color="xkcd:lightblue")
    ax.fill_between(neutron_x, neutron_y, -tail_base, where=where_tail, color=bg_grey)
    # ax.plot(gamma_x[where_total], gamma_y[where_total], color=bg_red, lw=3)
    ax.plot(gamma_x, gamma_y, color=bg_red, lw=2)
    ax.plot(neutron_x, neutron_y, color=bg_blue, lw=3)
    # ax.axhline(0, 0, 1)

    text_params = {
        "fontsize": fontsize - 4,
        "ha": "center",
        "va": "center"
    }
    ax.text(baseline_x, baseline_y, "Baseline", **text_params)
    ax.text(total_x, total_y, "Total Integral", **text_params)
    ax.text(tail_x, tail_y, "Tail Integral", **text_params)
    ax.text(t1, -baseline_adjust, r"$t_1$", fontsize=fontsize-6, ha="center", va="baseline")
    ax.text(t2, -baseline_adjust, r"$t_2$", fontsize=fontsize-6, ha="center", va="baseline")
    ax.text(t3, -baseline_adjust, r"$t_3$", fontsize=fontsize-6, ha="center", va="baseline")
    
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine_key in ["left", "right", "top", "bottom"]:
        ax.spines[spine_key].set_visible(False)
    # ax.set_xlabel("Time (ns)", fontsize=fontsize)
    # ax.set_ylabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax.set_ylim(-baseline_adjust, None)
    pass  # STUB

In [ ]:
def plot_figure_7c(ax: mpl.axes.Axes, fig: mpl.figure.Figure):
    exp_data = experiment_neutron_data["TB-26"]
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    valid_psds = psd_report[(psd_report["new_psd"] <= 1) & (psd_report["new_psd"] > 0)]
    height_vals = valid_psds["new_total"].values
    psd_vals = valid_psds["new_psd"].values

    # TODO get 2d histogram counts and use to color scatter points
    # counts, h_bins, psd_bins = np.histogram2d(height_vals, psd_vals, bins=(500, 100))
    *_, img = ax.hist2d(height_vals, psd_vals, bins=(500, 100), cmin=4, norm="log")
    cbar = fig.colorbar(img, ax=ax)
    # ax.scatter(height_vals, psd_vals, s=0.1, c=bg_blue, alpha=0.1)
    
    ax.set_xlim(0, 200000)
    ax.set_ylim(0, 0.5)
    ax.tick_params(labelsize=fontsize)
    ax.set_xlabel("Total integral (ADC channel ns x1000)", fontsize=fontsize)
    ax.set_ylabel("PSD", fontsize=fontsize)
    ax.xaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
    cbar.ax.tick_params(labelsize=fontsize)

In [ ]:
def plot_figure_7d(ax: mpl.axes.Axes, fig: mpl.figure.Figure):
    exp_data = experiment_neutron_data["TB-26"]
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    valid_psds = psd_report[(psd_report["new_psd"] <= 1) & (psd_report["new_psd"] > 0)]
    tail_vals = valid_psds["new_tail"].values
    total_vals = valid_psds["new_total"].values
    *_, img = ax.hist2d(tail_vals, total_vals, bins=250, cmin=1, norm="log")
    cbar = fig.colorbar(img, ax=ax)
    
    ax.set_xlim(0, 50000)
    ax.set_ylim(0, 200000)
    ax.tick_params(labelsize=fontsize)
    ax.set_xlabel("Tail integral (ADC channel ns x1000)", fontsize=fontsize)
    ax.set_ylabel("Total integral\n(ADC channel ns x1000)", fontsize=fontsize)
    ax.xaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
    ax.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
    cbar.ax.tick_params(labelsize=fontsize)

### Plot Creation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")
plot_figure_7a(ax)

In [ ]:
baseline_adjust = 2000
tail_base = 1000

fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")
plot_figure_7b(ax, 85, t1, t2, t3, baseline_adjust, tail_base)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_figure_7c(ax, fig)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_figure_7d(ax, fig)